# Week 06 - Boosting and SHAP-Guided Model Revision

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 07  
**Estimated time:** 3-4 hours

This notebook is self-contained except for SHAP. You will train boosted tree models, use SHAP values to understand what the model relies on, then change the modelling pipeline based on that explanation.


## What You Are Building

This week has five required functions:

1. `split_data(X, y, test_size, random_state)` - stratified train/test split.
2. `evaluate_classifiers(models, X_train, X_test, y_train, y_test)` - compare tree and boosting models numerically.
3. `compute_shap_importance(model, X_explain, feature_names)` - turn SHAP values into a global feature-importance table.
4. `evaluate_feature_subset(model, X_train, X_test, y_train, y_test, selected_features)` - retrain using only SHAP-selected features.
5. `make_benchmark_long(results, week, dataset, task_type, target, split)` - append results to the reusable benchmark format.

The point of SHAP here is not to decorate the notebook with an interpretability figure. You will use SHAP to decide which features to keep, retrain a smaller model, and test whether the explanation leads to a useful simplification.


In [ ]:
# Imports - keep this cell stable
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.25
TOP_K_FEATURES = 6


## SHAP Requirement

This exercise requires SHAP. If the import below fails, install it in your environment before continuing.

```bash
pip install shap
```

SHAP can be computationally heavier than ordinary model fitting. This notebook explains only the test set, which is small enough for the exercise.


In [ ]:
try:
    import shap
except ImportError as exc:
    raise ImportError("This notebook requires shap. Install it with: pip install shap") from exc


## Dataset

The Breast Cancer Wisconsin dataset is a binary classification problem with numeric tabular features. The target classes are malignant and benign diagnoses.


In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
# sklearn encodes malignant=0, benign=1. Keep that encoding and interpret it carefully.
y = data.target.rename("diagnosis")
feature_names = list(X.columns)
class_names = list(data.target_names)

print(X.shape)
print("Target classes:", dict(enumerate(class_names)))
display(X.head())
display(y.value_counts().sort_index().rename(index=dict(enumerate(class_names))))


## Task 1 - Stratified Split

Implement `split_data(X, y, test_size, random_state)`.

Return `(X_train, X_test, y_train, y_test)`. Use stratification so both classes are represented in train and test.


In [ ]:
def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Create a stratified train/test split for classification."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 1
X_train, X_test, y_train, y_test = split_data(X, y)

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert set(y_train.unique()) == set(y.unique())
assert set(y_test.unique()) == set(y.unique())
assert X_train.shape[1] == X.shape[1]

print("Task 1 passed")


## Task 2 - Evaluate Tree and Boosting Models

Implement `evaluate_classifiers(models, X_train, X_test, y_train, y_test)`.

Output columns:

- `model`
- `accuracy`
- `f1_macro`
- `roc_auc`
- `fit_time_sec`

Fit each model on the training set and evaluate on the test set. Sort by `f1_macro` descending.


In [ ]:
def evaluate_classifiers(models: dict, X_train, X_test, y_train, y_test) -> pd.DataFrame:
    """Fit classifiers and return comparable held-out test metrics."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 2
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "decision_tree_depth_3": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "random_forest_200": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "gradient_boosting_100": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
    "hist_gradient_boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

model_results = evaluate_classifiers(models, X_train, X_test, y_train, y_test)

assert isinstance(model_results, pd.DataFrame)
assert list(model_results.columns) == ["model", "accuracy", "f1_macro", "roc_auc", "fit_time_sec"]
assert len(model_results) == len(models)
assert model_results["f1_macro"].is_monotonic_decreasing
assert model_results[["accuracy", "f1_macro", "roc_auc", "fit_time_sec"]].notna().all().all()
assert model_results.loc[model_results["model"] == "gradient_boosting_100", "f1_macro"].iloc[0] > 0.85

print("Task 2 passed")
model_results


## Choose the Model to Explain

For SHAP, we will explain the `gradient_boosting_100` model even if another model slightly wins the benchmark. This keeps the interpretation tied to the boosting topic and uses a tree model that SHAP handles well.


In [ ]:
explained_model_name = "gradient_boosting_100"
explained_model = models[explained_model_name]
explained_model.fit(X_train, y_train)

y_pred = explained_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix: {explained_model_name}")
plt.tight_layout()
plt.show()


## Task 3 - Compute SHAP Global Importance

Implement `compute_shap_importance(model, X_explain, feature_names)`.

Use SHAP values to return a dataframe with columns:

- `feature`
- `mean_abs_shap`
- `rank`

Sort by `mean_abs_shap` descending. Rank starts at 1.

Implementation notes:

- Use `shap.TreeExplainer(model)` or `shap.Explainer(model, X_explain)`.
- Binary classifiers sometimes return a list or a 3D array of SHAP values. Your function should reduce this to one matrix of shape `(n_samples, n_features)` before taking absolute means.


In [ ]:
def compute_shap_importance(model, X_explain: pd.DataFrame, feature_names: list[str]) -> pd.DataFrame:
    """Compute global feature importance from mean absolute SHAP values."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 3
shap_importance = compute_shap_importance(explained_model, X_test, feature_names)

assert isinstance(shap_importance, pd.DataFrame)
assert list(shap_importance.columns) == ["feature", "mean_abs_shap", "rank"]
assert len(shap_importance) == len(feature_names)
assert shap_importance["mean_abs_shap"].ge(0).all()
assert shap_importance["mean_abs_shap"].is_monotonic_decreasing
assert shap_importance["rank"].tolist() == list(range(1, len(feature_names) + 1))

print("Task 3 passed")
shap_importance.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = shap_importance.head(12)
sns.barplot(data=plot_df, x="mean_abs_shap", y="feature", color="steelblue", ax=ax)
ax.set_title("Global SHAP importance for gradient boosting")
ax.set_xlabel("mean absolute SHAP value")
plt.tight_layout()
plt.show()


## Guided Analysis: Read the SHAP Summary Plot

The bar chart above is a compact ranking. The SHAP summary plot below carries more information: each point is a sample, horizontal position is SHAP impact, and colour is the feature value. Use this plot to decide whether the model relies on a few dominant features or a broad set of features.


In [ ]:
explainer = shap.TreeExplainer(explained_model)
raw_shap_values = explainer.shap_values(X_test)

# Handle SHAP output shape differences across versions/models.
if isinstance(raw_shap_values, list):
    shap_values_for_plot = raw_shap_values[-1]
elif getattr(raw_shap_values, "ndim", 0) == 3:
    shap_values_for_plot = raw_shap_values[:, :, -1]
else:
    shap_values_for_plot = raw_shap_values

shap.summary_plot(shap_values_for_plot, X_test, feature_names=feature_names, max_display=12)


## Task 4 - Retrain on SHAP-Selected Features

Implement `evaluate_feature_subset(model, X_train, X_test, y_train, y_test, selected_features)`.

Fit a fresh copy of the model using only `selected_features`, then return a one-row dataframe with columns. The `model` value should clearly identify the SHAP-selected subset, for example `gradient_boosting_shap_top6`:

- `model`
- `n_features`
- `accuracy`
- `f1_macro`
- `roc_auc`
- `fit_time_sec`

This is the modelling decision driven by SHAP: if the top SHAP features retain most of the score, the explanation led to a simpler model.


In [ ]:
def evaluate_feature_subset(model, X_train, X_test, y_train, y_test, selected_features: list[str]) -> pd.DataFrame:
    """Retrain a classifier on selected features and return held-out metrics."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 4
selected_features = shap_importance.head(TOP_K_FEATURES)["feature"].tolist()
subset_results = evaluate_feature_subset(
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
    X_train,
    X_test,
    y_train,
    y_test,
    selected_features,
)

assert isinstance(subset_results, pd.DataFrame)
assert list(subset_results.columns) == ["model", "n_features", "accuracy", "f1_macro", "roc_auc", "fit_time_sec"]
assert len(subset_results) == 1
assert subset_results["n_features"].iloc[0] == TOP_K_FEATURES
assert "shap_top" in subset_results["model"].iloc[0], "Name the model as a SHAP-selected feature subset"
assert subset_results[["accuracy", "f1_macro", "roc_auc"]].iloc[0].between(0, 1).all()

print("Task 4 passed")
print("Selected features:", selected_features)
subset_results


## Local SHAP Check

Before accepting the compact model, inspect one individual prediction. Choose a false negative if one exists, because missing a malignant case is clinically costly. If no false negative exists, inspect the first test sample.


In [ ]:
proba = explained_model.predict_proba(X_test)[:, 1]
local_df = X_test.copy()
local_df["true"] = y_test.values
local_df["pred"] = y_pred
local_df["p_benign"] = proba

false_negative = local_df[(local_df["true"] == 0) & (local_df["pred"] == 1)]
if len(false_negative) > 0:
    local_idx = false_negative.index[0]
    print("Inspecting false negative at index", local_idx)
else:
    local_idx = X_test.index[0]
    print("No false negative found; inspecting first test sample", local_idx)

row_pos = list(X_test.index).index(local_idx)
local_contrib = pd.DataFrame({
    "feature": feature_names,
    "feature_value": X_test.loc[local_idx].values,
    "shap_value": shap_values_for_plot[row_pos],
})
local_contrib["abs_shap"] = local_contrib["shap_value"].abs()
local_contrib = local_contrib.sort_values("abs_shap", ascending=False)
local_contrib.head(10)


## Task 5 - Benchmark Long Format

Implement `make_benchmark_long(results, week, dataset, task_type, target, split)`.

Convert model results into long benchmark format:

- `week`
- `dataset`
- `task_type`
- `target`
- `model`
- `metric`
- `score`
- `split`
- `notes`

Include numeric metric columns such as `accuracy`, `f1_macro`, and `roc_auc`. Do not include `fit_time_sec` or `n_features` as metrics.


In [ ]:
def make_benchmark_long(
    results: pd.DataFrame,
    week: str,
    dataset: str,
    task_type: str,
    target: str,
    split: str,
) -> pd.DataFrame:
    """Convert classifier results to the cumulative benchmark long format."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
subset_for_benchmark = subset_results.drop(columns=["n_features"])
all_results = pd.concat([model_results, subset_for_benchmark], ignore_index=True)
all_results = all_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)

benchmark_long = make_benchmark_long(
    all_results,
    week="W06",
    dataset="Breast Cancer Wisconsin",
    task_type="classification",
    target="diagnosis",
    split="75_25_stratified_random_state_42",
)

expected_cols = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
assert list(benchmark_long.columns) == expected_cols
assert {"accuracy", "f1_macro", "roc_auc"}.issubset(set(benchmark_long["metric"]))
assert "fit_time_sec" not in set(benchmark_long["metric"])
assert "n_features" not in set(benchmark_long["metric"])
assert benchmark_long["score"].between(0, 1).all()

print("Task 5 passed")
benchmark_long.head(12)


## Benchmark Wide View

The important comparison is now full boosted model vs SHAP-selected compact model, alongside the earlier baselines.


In [ ]:
benchmark_wide = (
    benchmark_long
    .pivot_table(
        index=["dataset", "task_type", "target", "metric", "split"],
        columns="model",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)
benchmark_wide.columns.name = None
benchmark_wide


## Reflection

Answer briefly, but concretely.

1. Which features did SHAP identify as most important? What does the summary plot say about high vs low values for at least one of them?
2. How much performance did the SHAP-selected compact model lose or gain compared with the full gradient boosting model?
3. Based on the local SHAP explanation, what feature values most influenced the inspected prediction?
4. Did SHAP change what you would do next with this model or dataset?


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Different Top-K
Repeat the SHAP-selected model with top 3, top 10, and top 15 features. Plot performance vs number of features.

### Track B - SHAP-Stability Check
Compute SHAP importance on two random halves of the test set. Are the top features stable?

### Track C - Compare Explanation Methods
Compare SHAP global importance with permutation importance. Where do they agree, and where do they disagree?


In [ ]:
# Optional challenge workspace
# Your code here
